# Model B, B1 — build fix 2: a stable likelihood (Oct 30, 2026) — EXPLORATORY

A build fix of B1 (numerics only), not a new version: `results/model_b_log.md`, entry 1. Fix 1: `notebooks/path1/pB1_fix1.ipynb`.

**Spec (as given):**

> Build fix 2, as approved: switch mixture_loglik to the stable form, add a regression test, recompute the old CL parametric CI, rerun the B1 recovery once (G6train, R = 2, seed 573), confirm pB1_params.csv is unchanged, log it under entry 1. Commit.

**The fix.** `model_b1.mixture_loglik` now writes B(a + s, b + t) / B(a, b) as (a)_s (b)_t / (a + b)_(s + t) and sums logs. The old `betaln(a + s, b + t) - betaln(a, b)` cancelled when rho is near 0 (a and b near 1e38) and reported too high a likelihood there. The likelihood itself is the same. Regression test: `tests/test_b1.py::test_mixture_likelihood_at_the_rho_limits` (closed forms at rho -> 0 and rho -> 1; the old form fails it).

**Choices.** Same code as fix 1 for everything else. "pB1_params.csv unchanged" is checked by refitting every pi, rho, q_sg and its group-bootstrap CI with the new form (same seed) and comparing with the file. Only the old CL rho parametric CI is rewritten.

In [1]:
import time

import numpy as np
import pandas as pd

from slider import config, design, model_b1, motion, sim, stop_states
from slider import slider_logs as sl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

runs, table = sl.load_runs("behavior")                  # roles={"train"}: G-train only, no G-val, Ring or CT
assert set(table["role"]) == {"train"}
att = model_b1.attempt_table(model_b1.step_state(stop_states.step_table(runs)))
params = pd.read_csv(config.PB1_PARAMS, float_precision="round_trip")    # keeps every digit: the file is rewritten

## 1. pi, rho, q_sg and their group-bootstrap CIs, refit with the stable form, vs. pB1_params.csv

In [2]:
fits, boots = {}, {}
for name in motion.INTERFACES:
    fits[name], boots[name] = model_b1.boot_attempts(att[att["interface"] == name])
refit = pd.DataFrame([{"interface": name, "parameter": par, "estimate": fits[name][i],
                       "ci_low": np.percentile(boots[name][:, i], 2.5), "ci_high": np.percentile(boots[name][:, i], 97.5)}
                      for name in motion.INTERFACES for i, par in enumerate(["pi", "rho", "q_sg"])])
cmp = params.merge(refit, on=["interface", "parameter"], suffixes=("_file", "_stable"))
cmp = cmp[["interface", "parameter", "ci_method"] + [f"{c}_{v}" for c in ["estimate", "ci_low", "ci_high"]
                                                    for v in ["file", "stable"]]]
parametric = cmp["ci_method"].str.startswith("parametric")
diff = np.abs(cmp[["estimate_file", "ci_low_file", "ci_high_file"]].to_numpy()
              - cmp[["estimate_stable", "ci_low_stable", "ci_high_stable"]].to_numpy())
diff[parametric.to_numpy(), 1:] = 0                      # the parametric CI is recomputed in step 2
cmp["max_abs_diff"] = diff.max(axis=1)
print("largest change, estimates and group-bootstrap CIs:", f"{diff.max():.2e}")
cmp.round(4)

largest change, estimates and group-bootstrap CIs: 1.70e-05


,interface,parameter,ci_method,estimate_file,estimate_stable,ci_low_file,ci_low_stable,ci_high_file,ci_high_stable,max_abs_diff
0,lab_v09,pi,group bootstrap,0.3404,0.3404,0.3031,0.3031,0.3787,0.3787,0.0
1,lab_v09,rho,group bootstrap,0.1701,0.1701,0.1069,0.1069,0.2420,0.2420,0.0
2,lab_v09,q_sg,group bootstrap,0.4768,0.4768,0.4173,0.4173,0.5311,0.5311,0.0
3,old_cl,pi,group bootstrap,0.1618,0.1618,0.0769,0.0769,0.3044,0.3043,0.0
4,old_cl,rho,"parametric (1,000 refits, seed 573); known lim...",0.1898,0.1898,0.0000,0.0000,1.0000,0.1898,0.0
5,old_cl,q_sg,group bootstrap,0.1406,0.1406,0.1207,0.1207,0.2059,0.2059,0.0
6,cc,pi,group bootstrap,0.3202,0.3202,0.2819,0.2819,0.3576,0.3576,0.0
7,cc,rho,group bootstrap,0.3935,0.3935,0.3476,0.3476,0.4420,0.4420,0.0
8,cc,q_sg,group bootstrap,0.5014,0.5014,0.4241,0.4241,0.5674,0.5674,0.0


## 2. Old CL rho: the parametric CI again (same draws as fix 1: 1,000 refits, seed 573)

In [3]:
old = att[att["interface"] == "old_cl"]
players_n = [a["n"].to_numpy() for _, a in old.groupby(["file", "player"])]
rng = np.random.default_rng(config.SEED)
pi0, rho0, q0 = params.set_index(["interface", "parameter"]).loc["old_cl"].loc[["pi", "rho", "q_sg"], "estimate"]
refits = np.array([model_b1.fit_attempts(model_b1.simulate_attempts(players_n, pi0, rho0, q0, rng)) for _ in range(1000)])
rho_ci = np.percentile(refits[:, 1], [2.5, 97.5])
i = (params["interface"] == "old_cl") & (params["parameter"] == "rho")
print(f"old CL rho = {rho0:.4f}; fix-1 parametric CI [{params.loc[i, 'ci_low'].item():.4f}, "
      f"{params.loc[i, 'ci_high'].item():.4f}]; fix-2 parametric CI [{rho_ci[0]:.4f}, {rho_ci[1]:.4f}] "
      f"(median {np.median(refits[:, 1]):.4f}; share of refits with rho < 1e-3: {(refits[:, 1] < 1e-3).mean():.3f})")
params.loc[i, ["ci_low", "ci_high"]] = rho_ci
params.loc[i, "ci_method"] = "parametric (1,000 refits, seed 573; build fix 2); known limitation: 14 players, 2 groups"
params.to_csv(config.PB1_PARAMS, index=False)

old CL rho = 0.1898; fix-1 parametric CI [0.0000, 1.0000]; fix-2 parametric CI [0.0000, 1.0000] (median 0.1855; share of refits with rho < 1e-3: 0.255)


## 3. Build check again: B1 on G6train, R = 2, seed 573 (estimates with the stable form)

In [4]:
G6 = design.design("G6train")
start = time.time()
rounds = design.simulate_rounds(G6, "B1", R=2, seed=573)              # pB1_params.csv (estimates unchanged)
sim_runs = [run for r in range(2) for run in design.copy_runs(G6, rounds, r, 573)]
est, used = model_b1.estimates(sim_runs)
seats_sim, _ = motion.motion_table(sim_runs)
lam = seats_sim.groupby("interface")[["resumes", "halted_s"]].sum()
est["lam_resume"] = lam["resumes"] / lam["halted_s"]
print(len(sim_runs), "rounds |", f"{(time.time() - start) / 60:.1f} min")
fix1 = pd.read_csv(config.PB1_RECOVERY)                               # build fix 1's recovery (commit 3424eb4)
rec = params.merge(est.reset_index().melt(id_vars="interface", var_name="parameter", value_name="re_estimate"),
                   on=["interface", "parameter"], how="left")
rec["pass"] = (rec["re_estimate"] >= rec["ci_low"]) & (rec["re_estimate"] <= rec["ci_high"])
rec = rec[["interface", "parameter", "estimate", "ci_low", "ci_high", "ci_method", "re_estimate", "pass"]].rename(
    columns={"estimate": "true (human fit)"})
rec["fix1_re_estimate"] = fix1["re_estimate"].to_numpy()
rec.to_csv(config.PB1_RECOVERY, index=False)
print(f"PASS {int(rec['pass'].sum())} of {len(rec)}  (fix 1: {int(fix1['pass'].sum())} of 15; v1: "
      f"{int(pd.read_csv(config.PB1_RECOVERY_V1)['pass'].sum())} of 15)")
rec.round(4)

488 rounds | 5.5 min
PASS 15 of 15  (fix 1: 14 of 15; v1: 12 of 15)


,interface,parameter,true (human fit),ci_low,ci_high,ci_method,re_estimate,pass,fix1_re_estimate
0,lab_v09,pi,0.3404,0.3031,0.3787,group bootstrap,0.3404,True,0.3404
1,lab_v09,rho,0.1701,0.1069,0.2420,group bootstrap,0.1786,True,0.1786
2,lab_v09,q_sg,0.4768,0.4173,0.5311,group bootstrap,0.4762,True,0.4762
3,lab_v09,q_E,0.4462,0.4063,0.4834,group bootstrap,0.4433,True,0.4433
4,lab_v09,lam_resume,1.7681,1.4773,2.0651,group bootstrap,1.7656,True,1.7656
5,old_cl,pi,0.1618,0.0769,0.3044,group bootstrap,0.1364,True,0.6259
6,old_cl,rho,0.1898,0.0000,1.0000,"parametric (1,000 refits, seed 573; build fix ...",0.0000,True,0.0000
7,old_cl,q_sg,0.1406,0.1207,0.2059,group bootstrap,0.1427,True,0.1450
8,old_cl,q_E,0.2538,0.2523,0.2547,group bootstrap,0.2541,True,0.2541
9,old_cl,lam_resume,1.7356,1.5600,2.1657,group bootstrap,1.7786,True,1.7786


**How to read it.** Step 1: every human estimate and group-bootstrap CI, refit with the stable form, against the file (the largest difference is printed). Step 2: the old CL rho parametric CI, from the same 1,000 simulated data sets as in fix 1, refit with the stable form. Step 3: B1 simulated on every G-train round (2 copies), re-estimated, PASS = inside the human 95% CI (old CL rho: the parametric CI). `fix1_re_estimate` is the fix-1 re-estimate on the same simulated runs (same seed, same parameters).